## ML_1050: Linear Regression

**Difficulty**: Medium | **TorchCode**: #40

### Problem
Implement linear regression three ways: (1) closed-form normal equation, (2) manual gradient descent without autograd, (3) PyTorch `nn.Linear` with autograd training loop. All three should converge to the same solution.

### Formula
$$\hat{\theta} = (X^T X)^{-1} X^T y \quad \text{(closed-form)}$$
$$\nabla_w \mathcal{L} = \frac{2}{N} X^T(Xw - y), \quad w \leftarrow w - \alpha \nabla_w \mathcal{L} \quad \text{(gradient descent)}$$

In [ ]:
import torch
import torch.nn as nn

class LinearRegression:
    def closed_form(self, X: torch.Tensor, y: torch.Tensor):
        N, D = X.shape
        X_aug = torch.cat([X, torch.ones(N, 1)], dim=1)
        theta = torch.linalg.lstsq(X_aug, y).solution
        return theta[:D].detach(), theta[D].detach()

    def gradient_descent(self, X: torch.Tensor, y: torch.Tensor,
                         lr: float = 0.01, steps: int = 1000):
        N, D = X.shape
        w = torch.zeros(D)
        b = torch.tensor(0.0)
        for _ in range(steps):
            error = X @ w + b - y
            w = w - lr * (2.0 / N) * (X.T @ error)
            b = b - lr * (2.0 / N) * error.sum()
        return w, b

    def nn_linear(self, X: torch.Tensor, y: torch.Tensor,
                  lr: float = 0.01, steps: int = 1000):
        N, D = X.shape
        layer = nn.Linear(D, 1)
        optimizer = torch.optim.SGD(layer.parameters(), lr=lr)
        loss_fn = nn.MSELoss()
        for _ in range(steps):
            optimizer.zero_grad()
            loss = loss_fn(layer(X).squeeze(-1), y)
            loss.backward()
            optimizer.step()
        return layer.weight.data.squeeze(0), layer.bias.data.squeeze(0)

In [ ]:
import torch

# --- Test 1: Closed-form returns correct shapes ---
torch.manual_seed(42)
X = torch.randn(50, 3)
y = X @ torch.tensor([2.0, -1.0, 0.5]) + 3.0 + torch.randn(50) * 0.01
model = LinearRegression()
w, b = model.closed_form(X, y)
assert w.shape == (3,) and b.shape == ()

# --- Test 2: Closed-form finds correct weights ---
torch.manual_seed(42)
true_w = torch.tensor([2.0, -1.0, 0.5])
X = torch.randn(100, 3)
y = X @ true_w + 3.0
w, b = model.closed_form(X, y)
assert torch.allclose(w, true_w, atol=1e-4)
assert torch.allclose(b, torch.tensor(3.0), atol=1e-4)

# --- Test 3: Gradient descent converges ---
torch.manual_seed(42)
w_gd, b_gd = model.gradient_descent(X, y, lr=0.05, steps=2000)
assert torch.allclose(w_gd, true_w, atol=0.1)
assert abs(b_gd.item() - 3.0) < 0.1

# --- Test 4: nn.Linear approach works ---
torch.manual_seed(42)
w_nn, b_nn = model.nn_linear(X, y, lr=0.05, steps=2000)
assert torch.allclose(w_nn, true_w, atol=0.1)
assert abs(b_nn.item() - 3.0) < 0.1

# --- Test 5: All three methods agree ---
torch.manual_seed(0)
X2 = torch.randn(200, 2)
y2 = X2 @ torch.tensor([1.5, -2.0]) + 1.0 + torch.randn(200) * 0.1
w_cf, b_cf = model.closed_form(X2, y2)
w_gd2, b_gd2 = model.gradient_descent(X2, y2, lr=0.05, steps=3000)
w_nn2, b_nn2 = model.nn_linear(X2, y2, lr=0.05, steps=3000)
assert torch.allclose(w_cf, w_gd2, atol=0.15)
assert torch.allclose(w_cf, w_nn2, atol=0.15)

# --- Test 6: Closed-form uses no autograd ---
X3 = torch.randn(30, 2)
y3 = X3 @ torch.tensor([1.0, 2.0]) + 0.5
w3, b3 = model.closed_form(X3, y3)
assert not w3.requires_grad

print('All tests passed!')